---

Import libs again

---

In [ ]:
import shutil
import logging
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
from scipy.stats import entropy
import cv2 as cv
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime

---

Config paths again

---

In [ ]:
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_PATH = REPORT_DIR / f"quality_cleaning_log_{RUN_TIMESTAMP}.log"

logger = logging.getLogger("quality_cleaning")
logger.setLevel(logging.INFO)
logger.handlers.clear()

# Console handler
console_handler = logging.StreamHandler()
console_handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
)
logger.addHandler(console_handler)

# File handler
file_handler = logging.FileHandler(LOG_PATH, encoding="utf-8")
file_handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
)
logger.addHandler(file_handler)

logger.info("=== Quality Cleaning Notebook Started ===")
logger.info(f"Run timestamp: {RUN_TIMESTAMP}")
logger.info(f"INTERIM_DIR: {INTERIM_DIR}")
logger.info(f"REPORT_DIR: {REPORT_DIR}")
logger.info(f"Log file: {LOG_PATH}")

print(f"Logging to: {LOG_PATH}")

---

Check for files extension and build an image inventory

---

In [ ]:
file_extensions = Counter()

for file_path in INTERIM_DIR.rglob("*"):
    if file_path.is_file():
        file_extensions[file_path.suffix.lower()] += 1

print(f"Total files found: {sum(file_extensions.values())}\n")

for extension, count in file_extensions.most_common():
    print(f"{extension if extension else '[no extension]'} : {count}")

---

Build an image inventory

---

In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png"
}

image_records = []

for image_path in INTERIM_DIR.rglob("*"):
    if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
        relative_path = image_path.relative_to(INTERIM_DIR)

        image_records.append({
            "source": relative_path.parts[0],
            "dataset": relative_path.parts[1] if len(relative_path.parts) > 1 else "unknown",
            "extension": image_path.suffix.lower(),
            "image_path": str(image_path)
        })

image_inventory = pd.DataFrame(image_records)

print(f"Total images discovered: {len(image_inventory)}")

image_inventory.head()

---

Check image resolution

---

In [ ]:
image_properties = []

for _, row in tqdm(image_inventory.iterrows(), total=len(image_inventory)):
    image_path = row["image_path"]

    image = cv.imread(image_path)

    if image is None:
        continue

    height, width, channels = image.shape

    image_properties.append({
        "source": row["source"],
        "dataset": row["dataset"],
        "image_path": image_path,
        "width": width,
        "height": height,
        "channels": channels
    })

image_quality_df = pd.DataFrame(image_properties)

print(f"Images analysed: {len(image_quality_df)}")

image_quality_df.head()

---

Image resolution summary

---

In [ ]:
print("Image Width Statistics")
print(image_quality_df["width"].describe())

print("\nImage Height Statistics")
print(image_quality_df["height"].describe())

print("\nMost Common Resolutions")

resolution_counts = (
    image_quality_df
    .groupby(["width", "height"])
    .size()
    .sort_values(ascending=False)
    .reset_index(name="count")
)

display(resolution_counts.head(20))

---

Check image blurness

---

In [ ]:
# === Blur Score Calculation ===

if "blur_score" in image_quality_df.columns:
    print("Blur score column already exists. Skipping blur calculation.")

else:
    blur_scores = []

    for _, row in tqdm(image_quality_df.iterrows(), total=len(image_quality_df)):
        image = cv.imread(row["image_path"])

        if image is None:
            blur_scores.append(None)
            continue

        gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)

        blur_score = cv.Laplacian(gray, cv.CV_64F).var()

        blur_scores.append(blur_score)

    image_quality_df["blur_score"] = blur_scores

    print("Blur scores calculated successfully.")

image_quality_df[["dataset", "blur_score"]].head()

---

Summarize blur score + threshold analysis

---

In [ ]:
# === Blur Score Analysis ===

if "blur_score" not in image_quality_df.columns:
    print("Blur score column not found. Run blur detection step first.")

else:
    print("=" * 60)
    print("Overall Blur Score Statistics")
    print("=" * 60)
    print(image_quality_df["blur_score"].describe())

    print("\nLowest Blur Scores")
    display(
        image_quality_df[
            ["dataset", "blur_score", "image_path"]
        ]
        .sort_values("blur_score")
        .head(20)
    )

    print("\nDataset Blur Statistics")

    dataset_blur = (
        image_quality_df
        .groupby("dataset")["blur_score"]
        .agg(["count", "mean", "median", "min", "max"])
        .sort_values("mean")
    )

    display(dataset_blur)

    print("\nBlur Threshold Analysis")

    thresholds = [10, 25, 50, 100, 200]

    blur_summary = []

    for threshold in thresholds:
        count = (image_quality_df["blur_score"] < threshold).sum()

        blur_summary.append({
            "Threshold": threshold,
            "Images": count,
            "Percentage (%)": round(count / len(image_quality_df) * 100, 2)
        })

    display(pd.DataFrame(blur_summary))

    plt.figure(figsize=(10, 5))
    plt.hist(image_quality_df["blur_score"], bins=50)

    plt.title("Blur Score Distribution")
    plt.xlabel("Variance of Laplacian")
    plt.ylabel("Number of Images")

    plt.show()

---

Channels/color-mode check

---

In [ ]:
# === Channels Check — Color Mode Consistency ===

if "channels" not in image_quality_df.columns:
    print("Channels column not found. Run channel detection step first.")

else:
    print("Channel Count Distribution (Overall)")
    print(image_quality_df["channels"].value_counts())

    print("\nChannel Count by Dataset")
    channel_by_dataset = (
        image_quality_df
        .groupby(["dataset", "channels"])
        .size()
        .unstack(fill_value=0)
    )
    display(channel_by_dataset)

    # Flag datasets that aren't uniformly 3-channel (the expected/normal case)
    mixed_datasets = channel_by_dataset[
        [c for c in [1, 4] if c in channel_by_dataset.columns]
    ]

    if mixed_datasets.empty or (mixed_datasets.sum(axis=1) == 0).all():
        print("All images are 3-channel (RGB) — no color mode inconsistency found")
    else:
        flagged = mixed_datasets[mixed_datasets.sum(axis=1) > 0]
        print("⚠️ Datasets with non-3-channel images found:")
        display(flagged)

---

Blank image check

---

In [ ]:
# === Blank Image Detection — Histogram Entropy, Brightness-Invariant ===

if "entropy" in image_quality_df.columns:
    print("Entropy column already exists. Skipping entropy calculation.")

else:
    entropy_scores = []

    for _, row in tqdm(image_quality_df.iterrows(), total=len(image_quality_df)):
        image = cv.imread(row["image_path"])
        if image is None:
            entropy_scores.append(None)
            continue

        gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
        hist = cv.calcHist([gray], [0], None, [256], [0, 256]).flatten()
        hist_norm = hist / hist.sum()
        hist_norm = hist_norm[hist_norm > 0]

        entropy_scores.append(entropy(hist_norm, base=2))

    image_quality_df["entropy"] = entropy_scores

    print("Entropy scores calculated successfully.")

image_quality_df.head()

---

Light exposure check

---

In [ ]:
# === Exposure Detection — Histogram Clipping ===

required_exposure_cols = {
    "dark_clip_pct",
    "bright_clip_pct",
    "mean_brightness"
}

if required_exposure_cols.issubset(image_quality_df.columns):
    print("Exposure metrics already exist. Skipping exposure calculation.")

else:
    exposure_records = []

    for _, row in tqdm(image_quality_df.iterrows(), total=len(image_quality_df)):
        image = cv.imread(row["image_path"])

        if image is None:
            exposure_records.append({
                "dark_clip_pct": None,
                "bright_clip_pct": None,
                "mean_brightness": None,
            })
            continue

        gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)

        hist = cv.calcHist([gray], [0], None, [256], [0, 256]).flatten()
        total_pixels = gray.size

        # % of pixels near pure black (0-10) or pure white (245-255)
        dark_clip_pct = hist[0:10].sum() / total_pixels * 100
        bright_clip_pct = hist[245:256].sum() / total_pixels * 100

        exposure_records.append({
            "dark_clip_pct": dark_clip_pct,
            "bright_clip_pct": bright_clip_pct,
            "mean_brightness": gray.mean(),
        })

    exposure_df = pd.DataFrame(exposure_records)

    image_quality_df["dark_clip_pct"] = exposure_df["dark_clip_pct"]
    image_quality_df["bright_clip_pct"] = exposure_df["bright_clip_pct"]
    image_quality_df["mean_brightness"] = exposure_df["mean_brightness"]

    print("Exposure metrics calculated.")

image_quality_df[["dataset", "dark_clip_pct", "bright_clip_pct", "mean_brightness"]].describe()

In [ ]:
print("Highest Bright-Clip % (most likely overexposed)")
display(
    image_quality_df[["dataset", "bright_clip_pct", "mean_brightness", "image_path"]]
    .sort_values("bright_clip_pct", ascending=False)
    .head()
)

print("\nHighest Dark-Clip % (most likely underexposed)")
display(
    image_quality_df[["dataset", "dark_clip_pct", "mean_brightness", "image_path"]]
    .sort_values("dark_clip_pct", ascending=False)
    .head()
)

print("\nExposure Stats by Dataset")
dataset_exposure = (
    image_quality_df
    .groupby("dataset")[["dark_clip_pct", "bright_clip_pct", "mean_brightness"]]
    .mean()
    .sort_values("bright_clip_pct", ascending=False)
)
display(dataset_exposure)

---

Generate a report for this section

---

In [ ]:
# === Final Report: Tag Quality Flags + Export ===

BLUR_THRESHOLD = 100
ENTROPY_THRESHOLD = 3.0
BRIGHT_CLIP_THRESHOLD = 90
DARK_CLIP_THRESHOLD = 85

def build_flags(row):
    flags = []
    if row["blur_score"] < BLUR_THRESHOLD:
        flags.append("low_blur")
    if row["entropy"] < ENTROPY_THRESHOLD:
        flags.append("low_entropy")
    if row["bright_clip_pct"] > BRIGHT_CLIP_THRESHOLD:
        flags.append("high_bright_clip")
    if row["dark_clip_pct"] > DARK_CLIP_THRESHOLD:
        flags.append("high_dark_clip")
    return ", ".join(flags) if flags else "none"

image_quality_df["quality_flags"] = image_quality_df.apply(build_flags, axis=1)

flagged_summary = (
    image_quality_df
    .assign(is_flagged=image_quality_df["quality_flags"] != "none")
    .groupby("dataset")["is_flagged"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "flagged", "count": "total"})
)
flagged_summary["flagged_pct"] = (flagged_summary["flagged"] / flagged_summary["total"] * 100).round(2)

print("Quality Flag Summary by Dataset")
display(flagged_summary)

report_path = REPORT_DIR / f"quality_cleaning_report_{RUN_TIMESTAMP}.csv"
image_quality_df.to_csv(report_path, index=False)
logger.info(f"Quality cleaning report saved: {report_path}")

print(f"\nReport saved: {report_path}")
print("interim/ left unchanged — no auto-removal this pass, all flagged extremes investigated and confirmed legitimate.")

---

End of quality cleaning

---